# Exploratory Data Analysis - CICIDS2017 Dataset

This notebook performs comprehensive exploratory data analysis on the CICIDS2017 intrusion detection dataset.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries imported successfully")

/usr/lib/python3/dist-packages/pytz/__init__.py:31: SyntaxWarning: invalid escape sequence '\s'
  match = re.match("^#\s*version\s*([0-9a-z]*)\s*$", line)


Libraries imported successfully


## 1. Load and Explore Raw Data

In [ ]:
# Load all CSV files
data_dir = Path('data/raw')
csv_files = sorted(data_dir.glob('*.csv'))

print(f"Found {len(csv_files)} CSV files:")
for f in csv_files:
    print(f"  - {f.name}")

In [ ]:
# Load and combine data
dfs = []
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file, encoding='utf-8', on_bad_lines='skip')
    except UnicodeDecodeError:
        df = pd.read_csv(csv_file, encoding='latin-1', on_bad_lines='skip')
    
    # Strip whitespace from column names
    df.columns = df.columns.str.strip()
    dfs.append(df)
    print(f"{csv_file.name}: {df.shape[0]} rows, {df.shape[1]} columns")

# Combine
df_raw = pd.concat(dfs, ignore_index=True)
print(f"\nCombined dataset: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns")

In [ ]:
# Display basic info
print("First few rows:")
df_raw.head()

In [ ]:
print("\nDataset Info:")
print(f"Shape: {df_raw.shape}")
print(f"\nColumns: {df_raw.columns.tolist()}")
print(f"\nData types:\n{df_raw.dtypes}")

## 2. Check for Missing and Infinite Values

In [ ]:
# Check for missing values
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw)) * 100

missing_df = pd.DataFrame({
    'Column': missing.index,
    'Missing_Count': missing.values,
    'Missing_Percent': missing_pct.values
})

missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Percent', ascending=False)
print(f"Columns with missing values: {len(missing_df)}")
print(missing_df)

In [ ]:
# Check for infinite values
numeric_cols = df_raw.select_dtypes(include=[np.number]).columns

inf_count = 0
inf_cols = []

for col in numeric_cols:
    col_inf = np.isinf(df_raw[col]).sum()
    if col_inf > 0:
        inf_count += col_inf
        inf_cols.append((col, col_inf))

print(f"Total infinite values: {inf_count}")
if inf_cols:
    print("\nColumns with infinite values:")
    for col, count in inf_cols:
        print(f"  {col}: {count}")

## 3. Check for Duplicates

In [ ]:
duplicates = df_raw.duplicated().sum()
print(f"Number of duplicate rows: {duplicates}")
print(f"Percentage: {(duplicates / len(df_raw)) * 100:.2f}%")

## 4. Class Distribution Analysis

In [ ]:
# Find label column
label_cols = ['Label', 'Attack_Type', 'Class']
label_col = None

for col in label_cols:
    if col in df_raw.columns:
        label_col = col
        break

if label_col:
    print(f"Using label column: {label_col}")
    print(f"\nUnique classes: {df_raw[label_col].nunique()}")
    print(f"\nClass distribution:")
    class_dist = df_raw[label_col].value_counts()
    print(class_dist)
    print(f"\nClass distribution (%)")
    print((class_dist / len(df_raw) * 100).round(2))

In [ ]:
# Plot class distribution
if label_col:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Bar plot
    class_dist.plot(kind='bar', ax=axes[0], color='skyblue', edgecolor='black')
    axes[0].set_title('Class Distribution (Count)', fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Count')
    axes[0].tick_params(axis='x', rotation=45)
    
    # Pie chart
    class_dist.plot(kind='pie', ax=axes[1], autopct='%1.1f%%')
    axes[1].set_title('Class Distribution (Percentage)', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('')
    
    plt.tight_layout()
    plt.show()

## 5. Feature Statistics

In [ ]:
# Basic statistics
print("Numeric Features Statistics:")
df_raw[numeric_cols].describe().T

In [ ]:
# Check feature ranges
print("Feature value ranges:")
ranges = pd.DataFrame({
    'Min': df_raw[numeric_cols].min(),
    'Max': df_raw[numeric_cols].max(),
    'Range': df_raw[numeric_cols].max() - df_raw[numeric_cols].min(),
    'Std': df_raw[numeric_cols].std()
})
ranges

## 6. Correlation Analysis

In [ ]:
# Compute correlation matrix
if len(numeric_cols) > 5:
    # Sample features for correlation (top 15 by variance)
    top_features = df_raw[numeric_cols].var().nlargest(15).index.tolist()
    corr_matrix = df_raw[top_features].corr()
else:
    corr_matrix = df_raw[numeric_cols].corr()

print(f"Correlation matrix shape: {corr_matrix.shape}")
print(f"Max correlation (non-diagonal): {(corr_matrix.values[~np.eye(len(corr_matrix), dtype=bool)].max()):.3f}")

In [ ]:
# Plot correlation heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Summary

In [ ]:
print("\n" + "="*60)
print("DATA SUMMARY")
print("="*60)
print(f"Total Samples: {len(df_raw):,}")
print(f"Total Features: {len(df_raw.columns)}")
print(f"Numeric Features: {len(numeric_cols)}")
print(f"Missing Values: {df_raw.isnull().sum().sum()}")
print(f"Duplicate Rows: {duplicates}")
print(f"Infinite Values: {inf_count}")
if label_col:
    print(f"Classes: {df_raw[label_col].nunique()}")
    imbalance_ratio = class_dist.max() / class_dist.min()
    print(f"Class Imbalance Ratio: {imbalance_ratio:.2f}x")
print("="*60)